# 03 — Tune Split 調參 → Test Split 定稿指標 + 失敗案例

**用途**:讀 M3 存在 Drive 的 keypoint cache(不需重跑推論)→ 在 **tune split**(10 falls + 13 adls)
網格搜尋規則引擎閾值 → 選出勝出設定 → 用同一組設定在 **test split**(20 falls + 27 adls,
n/s 兩模型各評一次)算出最終 precision/recall/F1 → 列出失敗案例(FP/FN)供下一步失敗分析。

**這本 notebook 全程只需 CPU**(規則引擎與模型推論解耦,只讀 cache,不碰 torch/ultralytics),
**可以選 CPU-only runtime**,不用消耗 GPU 額度。預期跑完全程約 5-10 分鐘(網格搜尋本身
約 2-3 分鐘)。

**調參設計(面試常被問的地方)**:
- **只調規則引擎階段仍可調的參數**:`kpt_conf_min`(關鍵點可見門檻)、`v_fall_enter`、
  `theta_lying_enter`(連動 `theta_upright_exit` 維持遲滯間距)、`t_confirm_fallen_s`。
  `model.conf`(偵測信心門檻)在 M3 extract 階段就已經烘進 cache,無法事後模擬,
  不在這個網格裡——要調它得重新跑一次 GPU extract。
- **網格刻意保持小**(每個參數 3 個候選值,共 3⁴=81 組):tune split 只有 23 支影片,
  組合太多容易在小樣本上過擬合出「剛好適合這 23 支」而非真正較好的設定。
- **選擇準則:recall 優先、precision ≥ 0.5 才列入候選**。跌倒偵測漏判的代價
  (沒人去查看、錯過黃金救援時間)高於多一次假警報,因此以 recall 為主要目標;
  但完全不設下限會讓「每一幀都報跌倒」這種退化解也拿到 recall=1.0,
  所以用 precision 下限擋掉明顯退化的組合。
- **n 模型負責調參,n/s 都在 test split 用同一組凍結閾值評估**:`yolo26n-pose`
  是本專案預設(速度優先),閾值以它的 tune 結果為準;`yolo26s-pose` 沿用同一組
  閾值在 test split 跑一次,作為模型對照(而非重新調參)。

跑完後把最後一格印出的 JSON 摘要貼回對話——我會依此把 `config.yaml` 的閾值
凍結、寫入 `eval/metrics.json`,並規劃下一步的失敗案例分析(frame strip + 特徵時序圖)。


In [ ]:
import os
if os.path.basename(os.getcwd()) != 'fall-detection-pose':
    if not os.path.exists('fall-detection-pose'):
        !git clone -q https://github.com/tun0000/fall-detection-pose.git
    %cd fall-detection-pose
!pip -q install -e "." pytest

# 同 notebook 01/02 的坑:pip install -e 之後 site.py 不會自動重新掃描 .pth,
# 這裡直接跑會 ModuleNotFoundError。reload(site) + 手動加 src/ 到 sys.path 雙重保險。
import importlib
import site
import sys

importlib.reload(site)
sys.path.insert(0, os.path.abspath("src"))

import fall_detection
print('fall_detection', fall_detection.__version__)


In [ ]:
# 規則引擎 + 調參邏輯單元測試(純 CPU,~10 秒):必須全綠
!python -m pytest -q


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/fall-detection-pose'
DATA_DIR = f'{DRIVE_ROOT}/data/urfd'
CACHE_DIR = f'{DRIVE_ROOT}/cache'
OUT_DIR = f'{DRIVE_ROOT}/outputs'
os.makedirs(OUT_DIR, exist_ok=True)
print('沿用 notebook 02 的 Drive 路徑:', DRIVE_ROOT)


In [ ]:
# 讀取 tune/test 名單、GT 事件、以及 yolo26n-pose 的全部 70 支 cache
from fall_detection.eval.splits import load_splits
from fall_detection.eval.ground_truth import load_gt_events
from fall_detection.io.cache import read_cache
from fall_detection.io.urfd import all_sequences, adl_sequences

splits = load_splits('eval/splits.yaml')
gt_falls = load_gt_events(f'{DATA_DIR}/urfall-cam0-falls.csv')

sequences = all_sequences()
adl_seqs = set(adl_sequences())
tune_seqs = splits['tune']['falls'] + splits['tune']['adls']
test_seqs = splits['test']['falls'] + splits['test']['adls']
print(f'tune: {len(tune_seqs)} 支, test: {len(test_seqs)} 支')

cache_n = {seq: read_cache(f'{CACHE_DIR}/yolo26n-pose/{seq}.parquet') for seq in sequences}
print(f'已載入 yolo26n-pose cache:{len(cache_n)} 支')


In [ ]:
# tune split 網格調參(純 CPU,3^4=81 組 x 23 支影片,約 2-3 分鐘)
from fall_detection.config import load_config
from fall_detection.eval.report import grid_search, make_param_grid, select_best

base_cfg = load_config('config.yaml')

grid = make_param_grid(
    kpt_conf_min_values=[0.25, 0.35, 0.45],
    v_fall_enter_values=[1.0, 1.5, 2.0],
    theta_lying_enter_values=[50, 55, 60],
    t_confirm_fallen_s_values=[0.7, 1.0, 1.3],
)
print(f'網格組合數:{len(grid)}')

results = grid_search(cache_n, tune_seqs, adl_seqs, gt_falls, base_cfg, grid, tol_s=0.5)
print(f'已在 tune split({len(tune_seqs)} 支影片)評估 {len(results)} 組設定')


In [ ]:
# 挑選規則:recall 優先、precision >= 0.5(避免「全部都報」的退化解)
# 詳見 eval/report.py 的 select_best docstring
best = select_best(results, min_precision=0.5)
print('=== 勝出設定(tune split) ===')
print('params:', best['params'])
print('tune metrics:', {k: v for k, v in best['metrics'].items() if k != 'per_video'})


In [ ]:
# 用勝出設定在 test split(47 支,n/s 各評一次)——這是 README 主表數字
from fall_detection.eval.matching import evaluate_videos
from fall_detection.eval.report import _apply_params, build_video_dicts

frozen_cfg = _apply_params(base_cfg, best['params'])

test_videos_n = build_video_dicts(cache_n, test_seqs, adl_seqs, gt_falls, frozen_cfg)
metrics_test_n = evaluate_videos(test_videos_n, tol_s=0.5)

cache_s = {seq: read_cache(f'{CACHE_DIR}/yolo26s-pose/{seq}.parquet') for seq in sequences}
test_videos_s = build_video_dicts(cache_s, test_seqs, adl_seqs, gt_falls, frozen_cfg)
metrics_test_s = evaluate_videos(test_videos_s, tol_s=0.5)

print('=== test split: yolo26n-pose ===')
print({k: v for k, v in metrics_test_n.items() if k != 'per_video'})
print('=== test split: yolo26s-pose ===')
print({k: v for k, v in metrics_test_s.items() if k != 'per_video'})


In [ ]:
# 列出 test split(yolo26n-pose)的 FP/FN 案例,供下一步失敗分析挑選
from fall_detection.eval.report import list_failure_cases

fail_n = list_failure_cases(metrics_test_n)
print(f"FP 案例({len(fail_n['fp_cases'])}):")
for c in fail_n['fp_cases']:
    print(' ', c['name'], c['fp_intervals'])
print(f"FN 案例({len(fail_n['fn_cases'])}):")
for c in fail_n['fn_cases']:
    print(' ', c['name'], c['fn_intervals'])


In [ ]:
import json

def _compact(m):
    return {k: v for k, v in m.items() if k != 'per_video'}

report = {
    'winning_params': best['params'],
    'tune_metrics': _compact(best['metrics']),
    'test_metrics_yolo26n_pose': _compact(metrics_test_n),
    'test_metrics_yolo26s_pose': _compact(metrics_test_s),
    'failure_cases_yolo26n_pose': {
        'fp': [{'name': c['name'], 'intervals': c['fp_intervals']} for c in fail_n['fp_cases']],
        'fn': [{'name': c['name'], 'intervals': c['fn_intervals']} for c in fail_n['fn_cases']],
    },
}

# 完整版(含每支影片的 preds/gts 明細)存 Drive 備查;貼回對話的是精簡版
with open(f'{OUT_DIR}/m4_full.json', 'w', encoding='utf-8') as f:
    json.dump(
        {'best': best, 'metrics_test_n': metrics_test_n, 'metrics_test_s': metrics_test_s},
        f, ensure_ascii=False, indent=2,
    )

print('=== M4 摘要(請貼回對話) ===')
print(json.dumps(report, ensure_ascii=False, indent=2))
